# Chapter 09 · Notebook 01: Meet weight decay

[Chapter guide](README.md) · [Previous chapter](../08_adam_optimizer/README.md) · [Course index](../README.md)

We have compared SGD and Adam. Now introduce one additional optimizer setting: weight decay. Begin with a small SGD experiment, then explore AdamW in later lessons.

Work through one lesson at a time with the mentoring conversation. Start in a fresh kernel. This notebook creates its own data and models and needs no earlier checkpoint.

## Contents

1. [Observe an update when the prediction-loss gradient is zero](#zero-gradient-decay)
2. [Trace decay for positive and negative weights](#decay-both-signs)
3. [Combine prediction error and decay in one update](#gradient-and-decay)
4. [Meet AdamW and compare its decay update](#meet-adamw)
5. [Inspect which memories include decay](#decay-in-memory)
6. [Combine prediction error and decay with AdamW](#adamw-with-error)

## Setup: one example and one weight

Use input `1` and target `2`. A model with weight `2` predicts the target exactly, so its mean squared prediction loss and the gradient from that loss are both zero.

Use `Linear(1, 1, bias=False)` to focus on one weight. The `bias=False` setting removes the bias parameter; this model has one learned scalar value. The input and target tensors each have shape `[1, 1]`.


In [ ]:
import torch

# One example lets us isolate the optimizer's effect on a single weight.
x = torch.tensor([[1.]])
y = torch.tensor([[2.]])
loss_fn = torch.nn.MSELoss()
learning_rate = 0.05

print("Input:", x.flatten().tolist())
print("Target:", y.flatten().tolist())
print("Learning rate:", learning_rate)


<a id="zero-gradient-decay"></a>
## Lesson 01: observe weight decay with a zero prediction-loss gradient

**Purpose:** separate the update driven by prediction error from the additional influence of weight decay.

Plain SGD without momentum or weight decay leaves a weight unchanged when its gradient is zero. A negative gradient, such as `-4`, would increase a weight at learning rate `0.05`: weight `2` would become `2.2`.

Weight decay adds an influence that pulls the weight toward zero. In SGD, the optimizer includes a contribution based on the current weight when choosing its update. We will explain the small numbers after observing the output.

Create two independent models at weight `2`. Use the same example and learning rate, with momentum disabled. Compare `weight_decay=0.0` and `weight_decay=0.1`. The latter is a demonstration setting, not a recommendation for other models.

For each model, calculate the prediction loss and its gradient normally before calling `step()`. Print the weight, prediction loss, and stored gradient before and after the update. The reported loss is always the mean squared prediction loss; we do not add a penalty to the printed loss calculation.

The code cell recreates both models and optimizers, so rerunning it repeats the experiment.

**Before running:** without weight decay, the weight should stay `2`. With weight decay, will it stay `2`, move toward zero, or move away from zero?

**My prediction and reason:** _write here_


In [ ]:
# Recreate the two runs so this lesson can be repeated independently.
results = {}
for decay in [0.0, 0.1]:
    model = torch.nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        model.weight.fill_(2.)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=learning_rate, momentum=0, weight_decay=decay
    )

    # Calculate the gradient from prediction error, using our usual sequence.
    optimizer.zero_grad(set_to_none=True)
    prediction_before = model(x)
    loss_before = loss_fn(prediction_before, y)
    loss_before.backward()
    weight_before = model.weight.item()
    gradient_before = model.weight.grad.item()

    # SGD considers its weight_decay setting when choosing the update.
    optimizer.step()
    with torch.no_grad():
        prediction_after = model(x)
        loss_after = loss_fn(prediction_after, y)

    results[decay] = {
        "weight_before": weight_before,
        "gradient_before": gradient_before,
        "weight_after": model.weight.item(),
        "loss_before": loss_before.item(),
        "loss_after": loss_after.item(),
    }
    print(f"Weight decay: {decay}")
    print("  Weight before step:", weight_before)
    print("  Prediction-loss gradient:", gradient_before)
    print("  Prediction loss before step:", loss_before.item())
    print("  Weight after step:", model.weight.item())
    print("  Prediction after step:", prediction_after.item())
    print("  Prediction loss after step:", loss_after.item())
    print("  Stored gradient after step:", model.weight.grad.item())


**My observation and explanation:** _write here_

Compare the stored prediction-loss gradients, then compare the weights after `step()`. Did a zero prediction-loss gradient prevent every parameter change? Which optimizer setting explains the difference?

Weight decay encourages smaller parameter magnitudes. It can be useful when choosing how to train a model, but it does not guarantee a lower prediction loss or better validation performance. This one-example experiment isolates its effect; it does not establish whether weight decay helps on unseen data.

Later lessons will examine the update size, a negative weight, and the interaction with fresh prediction-loss gradients before introducing AdamW.

### Reviewed result

With weight decay disabled, weight `2` stayed `2`. With `weight_decay=0.1` and learning rate `0.05`, it moved to approximately `1.99` even though the stored prediction-loss gradient stayed zero. Prediction loss increased slightly because the original prediction was already exact.



<a id="decay-both-signs"></a>
## Lesson 02: trace decay for positive and negative weights

**Purpose:** connect the observed change to small arithmetic steps and check what "toward zero" means for either sign.

For plain SGD with momentum disabled, multiply the current weight by the `weight_decay` setting, combine that contribution with the prediction-loss gradient, multiply by the learning rate, and subtract the result from the weight.

In this lesson, the prediction-loss gradient is zero. For weight `2`, the decay contribution is `0.1 * 2 = 0.2`; multiplying by learning rate `0.05` gives `0.01`. Subtracting this gives weight `1.99`.

For weight `-2`, the contribution is `0.1 * -2 = -0.2`, and the signed change to subtract is `-0.01`. Subtracting a negative number gives `-1.99`. Its numerical value increases, while its magnitude (distance from zero) decreases.

The setting `weight_decay=0.1` does not mean subtract `0.1` every time. The contribution depends on the current weight, and the learning rate controls the resulting change.

Create independent positive-weight and negative-weight examples. Each example uses input `1` and a target equal to its starting weight, so both begin with zero prediction loss and zero prediction-loss gradient. Their targets deliberately differ to isolate the same zero-gradient condition. This is an update demonstration, not a comparison of model quality.

Calculate the expected update using ordinary numbers, then apply a real SGD step and compare. This cell requires Setup and recreates both models each time it runs.

**Before running:** should the positive weight decrease while the negative weight increases? Will both have smaller magnitudes after the step?

**My prediction and reason:** _write here_


In [ ]:
# Check both signs using the same decay setting and learning rate.
decay_setting = 0.1
sign_results = {}
for starting_weight in [2.0, -2.0]:
    sign_model = torch.nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        sign_model.weight.fill_(starting_weight)

    # Match the starting prediction so prediction error contributes zero gradient.
    sign_target = torch.tensor([[starting_weight]])
    sign_optimizer = torch.optim.SGD(
        sign_model.parameters(), lr=learning_rate, momentum=0,
        weight_decay=decay_setting
    )
    sign_optimizer.zero_grad(set_to_none=True)
    sign_loss = loss_fn(sign_model(x), sign_target)
    sign_loss.backward()
    prediction_gradient = sign_model.weight.grad.item()

    # Trace the update with ordinary numbers; these lines do not modify the model.
    decay_contribution = decay_setting * starting_weight
    combined_direction = prediction_gradient + decay_contribution
    signed_change_to_subtract = learning_rate * combined_direction
    expected_weight = starting_weight - signed_change_to_subtract

    # Perform one real optimizer update for comparison with our calculation.
    sign_optimizer.step()
    actual_weight = sign_model.weight.item()
    sign_results[starting_weight] = actual_weight
    print(f"Starting weight: {starting_weight:+.2f}")
    print(f"  Prediction-loss gradient: {prediction_gradient:+.4f}")
    print(f"  Decay contribution: {decay_contribution:+.4f}")
    print(f"  Signed change to subtract: {signed_change_to_subtract:+.4f}")
    print(f"  Expected weight: {expected_weight:+.4f}")
    print(f"  Actual weight: {actual_weight:+.4f}")
    print(f"  Magnitude before: {abs(starting_weight):.4f}")
    print(f"  Magnitude after: {abs(actual_weight):.4f}")
    print("  Calculation agrees:", abs(actual_weight - expected_weight) < 1e-6)


**My observation and explanation:** _write here_

Both weights have smaller magnitudes after the step. The positive weight decreases; the negative weight increases. Moving toward zero depends on the weight's sign.

These calculations isolate decay by using zero prediction-loss gradients and no momentum. When the prediction-loss gradient is nonzero, both influences contribute to the update; the final weight need not move toward zero. We will observe that interaction next.

### Reviewed result

The arithmetic matched SGD: positive weight `2` became `1.99`, and negative weight `-2` became `-1.99`. Both magnitudes decreased from `2` to `1.99`. The decay contribution pulls either sign toward zero; the numerical value of a negative weight increases as it approaches zero.



<a id="gradient-and-decay"></a>
## Lesson 03: combine prediction error and decay in one update

**Purpose:** observe that the final weight can move away from zero even though the decay contribution pulls toward zero.

Use input `1` again, but change the target to `4` for this lesson. Start independent models at weight `2`, with no bias. Each model predicts `2`, so its squared prediction error is `4`, and backward produces prediction-loss gradient `-4`.

Compare plain SGD with momentum disabled, learning rate `0.05`, and weight decay settings `0` and `0.1`. With decay enabled, the current weight contributes `+0.2`. Combine this with gradient `-4` to get direction `-3.8`. Subtracting a negative direction increases the weight: `2 - 0.05 * (-3.8) = 2.19`.

The decay contribution opposes the increase driven by prediction error. Here it reduces that increase, but it does not reverse it. The final update reflects both contributions.

The code creates the lesson's target separately; Setup's original target remains available for earlier lessons. It recreates both models and optimizers each time, so the cell can be repeated after running Setup. Report prediction loss without adding a penalty to the displayed MSE.

**Before running:** without decay, the weight should become `2.2`. With decay, should it become `2.19`? Is either updated weight closer to zero than the starting weight?

**My prediction and reason:** _write here_


In [ ]:
# This lesson needs a prediction error, so use a new target without changing y.
interaction_target = torch.tensor([[4.]])
interaction_results = {}
for interaction_decay in [0.0, 0.1]:
    interaction_model = torch.nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        interaction_model.weight.fill_(2.)
    interaction_optimizer = torch.optim.SGD(
        interaction_model.parameters(), lr=learning_rate, momentum=0,
        weight_decay=interaction_decay
    )

    # Calculate the fresh gradient from the prediction error.
    interaction_optimizer.zero_grad(set_to_none=True)
    prediction_before = interaction_model(x)
    loss_before = loss_fn(prediction_before, interaction_target)
    loss_before.backward()
    weight_before = interaction_model.weight.item()
    prediction_gradient = interaction_model.weight.grad.item()

    # Reconstruct plain SGD's direction, including its decay contribution.
    decay_contribution = interaction_decay * weight_before
    combined_direction = prediction_gradient + decay_contribution
    expected_weight = weight_before - learning_rate * combined_direction

    # Apply the real update, then measure prediction loss at the new weight.
    interaction_optimizer.step()
    with torch.no_grad():
        prediction_after = interaction_model(x)
        loss_after = loss_fn(prediction_after, interaction_target)
    actual_weight = interaction_model.weight.item()
    interaction_results[interaction_decay] = actual_weight

    print(f"Weight decay: {interaction_decay}")
    print(f"  Weight before step: {weight_before:.4f}")
    print(f"  Prediction-loss gradient: {prediction_gradient:+.4f}")
    print(f"  Decay contribution: {decay_contribution:+.4f}")
    print(f"  Combined direction: {combined_direction:+.4f}")
    print(f"  Expected weight: {expected_weight:.4f}")
    print(f"  Actual weight: {actual_weight:.4f}")
    print(f"  Weight increase: {actual_weight - weight_before:.4f}")
    print(f"  Prediction loss before: {loss_before.item():.6f}")
    print(f"  Prediction loss after: {loss_after.item():.6f}")
    print("  Stored prediction-loss gradient:", interaction_model.weight.grad.item())
    print("  Calculation agrees:", abs(actual_weight - expected_weight) < 1e-6)


**My observation and explanation:** _write here_

Compare the weight increases. Both models moved away from zero because the negative prediction-loss gradient outweighed the decay contribution. The decay contribution still opposed the increase.

The reported `.grad` comes from our prediction-loss calculation. In this SGD experiment, `step()` uses an additional decay contribution internally without replacing that stored gradient.

Prediction loss after one update does not establish which setting generalizes better. Weight decay is a training setting to evaluate using validation data, alongside other settings such as the learning rate.

Next, observe how Adam and AdamW handle weight decay differently. Keep the idea of prediction-error and decay contributions separate as we introduce that comparison.

### Reviewed result

Both runs started with prediction-loss gradient `-4`. Without decay, weight `2` became `2.20`; with decay `0.1`, the combined direction was `-3.8` and the weight became `2.19`. The prediction-error contribution increased the weight by `0.20`, while decay opposed that increase by `0.01`. The full manual prediction, loss, gradient, and SGD calculation was also reviewed in the mentoring conversation.



<a id="meet-adamw"></a>
## Lesson 04: meet AdamW and compare its decay update

**Purpose:** observe that Adam and AdamW can produce different changes with the same weight-decay and learning-rate settings.

Recall that Adam uses running gradient and squared-gradient averages to choose its updates. In our installed PyTorch 2.6, `torch.optim.Adam` adds its weight-decay contribution to the prediction-loss gradient before processing that direction through its memories and size scaling.

`torch.optim.AdamW` applies decay as a separate shrink of the parameter. Its gradient memories and adaptive update use the prediction-loss gradient without adding that decay contribution. This separation is called **decoupled weight decay**. Keep the explanation in words for now; we will inspect the actual memories in a later lesson.

Return to input `1`, target `2`, and starting weight `2`, with no bias. All runs begin with exact predictions, zero prediction loss, and zero prediction-loss gradient. Use fresh optimizers with learning rate `0.05`, default memory settings, and either no decay or decay `0.1`.

Compare three independent runs: Adam without decay, Adam with decay, and AdamW with decay. Set `weight_decay` explicitly for every optimizer so their defaults do not determine the comparison. This cell recreates all three runs whenever it executes and requires only Setup.

We are observing an update mechanism, not selecting the best optimizer for unseen data. A one-example prediction loss cannot establish a general optimizer ranking. These settings are for demonstration.

**Before running:** will Adam and AdamW necessarily shrink the weight by the same amount just because both have `weight_decay=0.1` and learning rate `0.05`?

**My prediction and reason:** _write here_


In [ ]:
# Return to an exact starting prediction to isolate decay from prediction error.
adam_decay_target = torch.tensor([[2.]])
adam_decay_runs = {}
for case_name, optimizer_class, case_decay in [
    ("Adam, no decay", torch.optim.Adam, 0.0),
    ("Adam, decay 0.1", torch.optim.Adam, 0.1),
    ("AdamW, decay 0.1", torch.optim.AdamW, 0.1),
]:
    case_model = torch.nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        case_model.weight.fill_(2.)

    # Explicit settings make this a controlled first-update comparison.
    case_optimizer = optimizer_class(
        case_model.parameters(), lr=learning_rate, weight_decay=case_decay,
        betas=(0.9, 0.999), eps=1e-8
    )
    case_optimizer.zero_grad(set_to_none=True)
    case_loss = loss_fn(case_model(x), adam_decay_target)
    case_loss.backward()
    case_weight_before = case_model.weight.item()
    case_prediction_gradient = case_model.weight.grad.item()

    # Each optimizer uses its own handling of decay when choosing the update.
    case_optimizer.step()
    with torch.no_grad():
        case_prediction_after = case_model(x)
        case_loss_after = loss_fn(case_prediction_after, adam_decay_target)
    case_weight_after = case_model.weight.item()

    # Keep the separate objects so a later lesson can inspect their memories.
    adam_decay_runs[case_name] = {
        "model": case_model,
        "optimizer": case_optimizer,
        "weight_after": case_weight_after,
    }
    print(case_name)
    print(f"  Weight before step: {case_weight_before:.6f}")
    print(f"  Prediction-loss gradient: {case_prediction_gradient:.6f}")
    print(f"  Prediction loss before step: {case_loss.item():.6f}")
    print(f"  Weight after step: {case_weight_after:.6f}")
    print(f"  Weight decrease: {case_weight_before - case_weight_after:.6f}")
    print(f"  Prediction loss after step: {case_loss_after.item():.6f}")
    print(f"  Stored gradient after step: {case_model.weight.grad.item():.6f}")


**My observation and explanation:** _write here_

Compare the two decay-enabled runs. The prediction-loss gradients were identical, but the weights changed by different amounts. The same setting name and value does not imply identical handling inside different optimizers.

For this fresh, zero-gradient example, AdamW applies the separate shrink `0.05 * 0.1 * 2 = 0.01`. Adam processes the decay contribution through its adaptive update instead. Neither method promises better validation performance solely because of its first-step change.

Next, inspect the stored averages to see which optimizer's memories include the decay contribution. Inspection will not perform another update.

### Reviewed result

After one fresh update from weight `2` with zero prediction-loss gradient, Adam without decay stayed at `2`, Adam with decay `0.1` moved to `1.95`, and AdamW with decay `0.1` moved to `1.99`. Adam processed the added direction `+0.2` through its adaptive update; AdamW applied the separate shrink of `0.01`. The stored prediction-loss gradients stayed zero in all three runs.



<a id="decay-in-memory"></a>
## Lesson 05: inspect which memories include decay

**Purpose:** connect the different decay updates to the information stored in Adam's and AdamW's running averages.

Read the three independent model and optimizer pairs from Lesson 04. Each optimizer has performed exactly one update. Their prediction-loss gradients were all zero.

Recall the state entry names from Chapter 08: `step` counts updates, `exp_avg` stores the running gradient average, and `exp_avg_sq` stores the running squared-gradient average. The optimizer's `state` dictionary is indexed by the parameter object.

Adam with decay processed direction `+0.2`, including the weight-decay contribution. With fresh memories and the default settings, its first average is `0.1 * 0.2 = 0.02`. Its first squared average is `0.001 * (0.2 ** 2) = 0.00004`.

AdamW used prediction-loss gradient `0` for its averages, so both first averages are zero. Its separate shrink still changed the weight. Adam without decay also processed zero and has zero averages.

This cell only reads state. It does not call `backward()`, `zero_grad()`, or `step()`, so it can be repeated without changing the parameters, memories, or update counts. Run Lesson 04 first; Setup alone does not create these optimizer pairs.

**Before running:** which case should have a nonzero stored gradient average even though its prediction-loss `.grad` is zero?

**My prediction and reason:** _write here_


In [ ]:
# Inspect the existing objects from Lesson 04 without performing another update.
for case_name, saved_run in adam_decay_runs.items():
    stored_model = saved_run["model"]
    stored_optimizer = saved_run["optimizer"]
    parameter = stored_model.weight

    # Optimizer state belongs to this particular parameter object.
    memory = stored_optimizer.state[parameter]
    current_gradient = parameter.grad
    gradient_value = None if current_gradient is None else current_gradient.item()

    print(case_name)
    print("  Current prediction-loss gradient:", gradient_value)
    print("  State entry names:", list(memory.keys()))
    print("  Updates recorded:", int(memory["step"].item()))
    print(f"  Remembered gradient average: {memory['exp_avg'].item():.8f}")
    print(f"  Remembered squared-gradient average: {memory['exp_avg_sq'].item():.8f}")
    print(f"  Weight still: {parameter.item():.6f}")


**My observation and explanation:** _write here_

Adam with decay stored nonzero averages even though its prediction-loss gradient was zero. AdamW's averages stayed zero, while its parameter still shrank. This separates the stored prediction-loss gradient, the optimizer's running memories, and the current parameter value.

Zero averages do not mean that AdamW cannot learn. This experiment deliberately started with zero prediction error and fresh history. When prediction error produces a nonzero gradient, AdamW uses that gradient in its memories and adaptive update, as well as applying its separate decay.

Next, give both optimizers a nonzero prediction-loss gradient and inspect how the gradient update and decay interact.

### Reviewed result

Adam with decay stored first averages `0.02` and `0.00004`, reflecting its combined direction `+0.2`. AdamW with decay stored zero averages from the zero prediction-loss gradient, while its weight had still changed to `1.99`. Reading these states left all update counts at `1` and all weights unchanged. A nonzero prediction-loss gradient would make AdamW's fresh averages nonzero.



<a id="adamw-with-error"></a>
## Lesson 06: combine prediction error and decay with AdamW

**Purpose:** observe AdamW learning from a nonzero prediction-loss gradient while applying decay separately.

Create fresh Adam and AdamW runs at weight `2`, with no bias, input `1`, and target `4`. Both start with prediction `2`, squared prediction loss `4`, and prediction-loss gradient `-4`. Use learning rate `0.05`, decay `0.1`, and default memory settings.

For Adam, the decay contribution `+0.2` combines with prediction-loss gradient `-4` to give direction `-3.8` for its adaptive update. Its fresh averages should be `0.1 * (-3.8) = -0.38` and `0.001 * (-3.8) ** 2 = 0.01444`.

For AdamW, the gradient averages use `-4` without adding decay. Its fresh averages should be `-0.4` and `0.016`. The separate decay reduces the starting weight by `0.05 * 0.1 * 2 = 0.01`. The first adaptive update increases the weight by approximately `0.05`, producing final weight approximately `2.04`.

Adam's first adaptive update also increases its weight by approximately `0.05`, producing weight approximately `2.05`. This first-step behavior depends on fresh history and the current settings; it does not mean decay never affects Adam, or that future updates always have the same size.

These are new objects. The zero-gradient examples and histories from Lessons 04–05 remain separate. The cell requires Setup and recreates both runs whenever executed. Printed losses are prediction MSE, without adding a penalty to the displayed loss.

**Before running:** will AdamW's averages remain zero? Can its final weight increase even though its separate decay operation shrinks the weight?

**My prediction and reason:** _write here_


In [ ]:
# A different target creates prediction error at the starting weight.
error_target = torch.tensor([[4.]])
error_runs = {}
for error_name, error_optimizer_class in [
    ("Adam", torch.optim.Adam),
    ("AdamW", torch.optim.AdamW),
]:
    error_model = torch.nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        error_model.weight.fill_(2.)
    error_optimizer = error_optimizer_class(
        error_model.parameters(), lr=learning_rate, weight_decay=0.1,
        betas=(0.9, 0.999), eps=1e-8
    )

    # The prediction-loss calculation is identical for both optimizers.
    error_optimizer.zero_grad(set_to_none=True)
    prediction_before = error_model(x)
    loss_before = loss_fn(prediction_before, error_target)
    loss_before.backward()
    gradient_before = error_model.weight.grad.item()
    weight_before = error_model.weight.item()

    # Perform exactly one update with each optimizer's own decay handling.
    error_optimizer.step()
    with torch.no_grad():
        prediction_after = error_model(x)
        loss_after = loss_fn(prediction_after, error_target)
    memory = error_optimizer.state[error_model.weight]
    error_runs[error_name] = {"model": error_model, "optimizer": error_optimizer}

    print(error_name)
    print(f"  Weight before step: {weight_before:.6f}")
    print(f"  Prediction-loss gradient: {gradient_before:.6f}")
    print(f"  Remembered gradient average: {memory['exp_avg'].item():.6f}")
    print(f"  Remembered squared-gradient average: {memory['exp_avg_sq'].item():.6f}")
    print("  Updates recorded:", int(memory["step"].item()))
    print(f"  Weight after step: {error_model.weight.item():.6f}")
    print(f"  Weight increase: {error_model.weight.item() - weight_before:.6f}")
    print(f"  Prediction loss before step: {loss_before.item():.6f}")
    print(f"  Prediction loss after step: {loss_after.item():.6f}")


**My observation and explanation:** _write here_

Both optimizers receive the same prediction-loss gradient `-4`, yet their averages differ because Adam includes the decay contribution and AdamW keeps it separate.

AdamW's nonzero averages show that it learns from prediction error. Its final weight increases because the adaptive gradient update outweighs the separate shrink. Decay does not guarantee that the final weight always moves toward zero.

Compare these fresh results with Lessons 04–05, where the prediction-loss gradient was zero. Keep the first-step demonstration distinct from claims about later behavior or optimizer quality on unseen data.

After reviewing the result, revisit the chapter's main distinctions: prediction-loss gradients, optimizer history, decay contributions, and the final parameter update. Select training settings using appropriate validation results.

<a id="chapter-09-review"></a>
## Chapter 09 recap

All six lessons have been run and reviewed in the mentoring conversation. The final understanding check identified validation loss as the basis for selecting optimizer settings.

- Plain SGD with no momentum or decay leaves a weight unchanged when its prediction-loss gradient is zero.
- The decay contribution pulls a positive or negative weight toward zero. The size of the change depends on the weight, learning rate, and decay setting; `weight_decay=0.1` does not mean subtract `0.1` each time.
- The final parameter update can move away from zero when the prediction-error contribution outweighs decay.
- In the PyTorch 2.6 Adam examples, decay is added to the prediction-loss gradient before updating the running averages and choosing the adaptive update.
- AdamW applies decay separately. Its averages use the prediction-loss gradient without adding the decay contribution.
- The stored `.grad`, optimizer memories, and current parameter value are different pieces of information. Reading optimizer state performs no additional update.
- A zero prediction-loss gradient and fresh history isolated decay. AdamW still learns from nonzero prediction-loss gradients, updating its memories and applying an adaptive update as well as decay.
- Our approximately learning-rate-sized Adam first updates depended on fresh history and the observed directions. Later behavior depends on evolving gradients and history.
- Weight decay does not guarantee lower prediction loss or better validation performance. Compare optimizer and setting combinations using appropriate validation results and consistent training budgets. Keep the best validation checkpoint and use independent test data after selection.

### Recorded SGD calculations

With learning rate `0.05`, decay `0.1`, and zero prediction-loss gradient, weight `2` became `1.99` and weight `-2` became `-1.99`.

For input `1`, target `4`, and starting weight `2`, prediction loss was `4` and its weight gradient was `-4`. Plain SGD without decay produced weight `2.20`. With decay, the contribution `+0.2` gave combined direction `-3.8` and final weight `2.19`; the resulting prediction loss was `3.2761`. The full manual prediction, loss, gradient, and update calculation was reviewed alongside commented PyTorch code.

### Recorded first Adam and AdamW updates

Both comparisons used starting weight `2`, input `1`, learning rate `0.05`, fresh optimizer history, and default average settings. Printed losses were prediction MSE.

| Target and prediction-loss gradient | Optimizer | Decay setting | Weight after first update |
|---|---|---:|---:|
| Target 2; gradient 0 | Adam | 0 | 2.00 |
| Target 2; gradient 0 | Adam | 0.1 | 1.95 |
| Target 2; gradient 0 | AdamW | 0.1 | 1.99 |
| Target 4; gradient -4 | Adam | 0.1 | 2.05 |
| Target 4; gradient -4 | AdamW | 0.1 | 2.04 |

For target `2`, Adam with decay stored first averages `0.02` and `0.00004`; AdamW stored zero averages while still shrinking its weight. For target `4`, Adam stored `-0.38` and `0.01444`, while AdamW stored `-0.4` and `0.016`.

AdamW's final increase in the target-4 example combined an approximately `0.05` adaptive increase with a `0.01` separate decay decrease. These first-step results explain the mechanisms; they do not establish a universal optimizer ranking.

Next, introduce learning-rate schedules: changing the learning-rate setting as training progresses.

## Documentation

- [PyTorch: SGD and its weight_decay setting](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

- [PyTorch: Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)
- [PyTorch: AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)
